# Episode 9 — USB webcam and OpenCV basics

**OpenCV** is the standard library for computer vision. On PYNQ it runs on the ARM cores,
and pictures come from a USB webcam.

**Hardware:** USB webcam on connector **J3** with jumpers **J6 and J7** fitted (USB host mode),
and an HDMI monitor for the live parts.

An OpenCV image is a NumPy array: `height × width × 3`, one byte per colour, in **B, G, R** order.

In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_BGR
import numpy as np
import cv2
import time
%matplotlib inline
import matplotlib.pyplot as plt

base = BaseOverlay("base.bit")

cam = cv2.VideoCapture(0, cv2.CAP_V4L2)
cam.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
print("Webcam open:", cam.isOpened())

def grab():
    """One webcam picture as 640x480 BGR (some webcams ignore the size request)."""
    ok, img = cam.read()
    if ok and img.shape[:2] != (480, 640):
        img = cv2.resize(img, (640, 480))
    return ok, img

If the webcam is not open, check J6/J7 and the cable, then run `!ls /dev/video*` to see
whether Linux found it.

## 1. Take a picture

Matplotlib expects **R, G, B**, so convert before showing.

In [ ]:
ok, img = grab()
print(img.shape, img.dtype)
cv2.imwrite("snapshot.jpg", img)          # saved next to this notebook
plt.figure(figsize=(6, 4.5))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## 2. Colour spaces

* **Grey:** one brightness value per pixel.
* **HSV:** hue (which colour, 0–179 in OpenCV), saturation (how strong), value (how bright).
  Picking objects by colour is much easier in HSV than in BGR.

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

fig, ax = plt.subplots(1, 4, figsize=(14, 3))
ax[0].imshow(gray, cmap="gray"); ax[0].set_title("grey")
for i, name in enumerate(["hue", "saturation", "value"]):
    ax[i + 1].imshow(hsv[:, :, i], cmap="gray"); ax[i + 1].set_title(name)
for a in ax:
    a.axis("off")
plt.show()

## 3. Blur and threshold

A threshold turns a grey picture into black and white: pixels brighter than the limit
become white. Blurring first removes noise.

In [ ]:
blur = cv2.GaussianBlur(gray, (7, 7), 0)
_, fixed = cv2.threshold(blur, 128, 255, cv2.THRESH_BINARY)
_, otsu = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)   # picks the limit itself

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, im, title in zip(ax, [blur, fixed, otsu], ["blurred", "threshold 128", "Otsu threshold"]):
    a.imshow(im, cmap="gray"); a.set_title(title); a.axis("off")
plt.show()

## 4. Webcam to HDMI, live

Press **button 0** to stop (or wait 20 seconds). The frame rate is on the 7-segment display.

In [ ]:
hdmi_out = base.video.hdmi_out
hdmi_out.configure(VideoMode(640, 480, 24), PIXEL_BGR)
hdmi_out.start();

frames, t0 = 0, time.time()
while time.time() - t0 < 20 and not base.buttons[0].read():
    ok, img = grab()
    if not ok:
        break
    out = hdmi_out.newframe()
    out[:] = img
    hdmi_out.writeframe(out)
    frames += 1
    if frames % 15 == 0:
        base.seven_seg.show_number(round(frames / (time.time() - t0), 1))
print("{:.1f} frames per second".format(frames / (time.time() - t0)))

## 5. What colour is my object?

Hold a brightly coloured object (a blue cap, a red ball) in the middle of the picture
and run this cell. It prints the object's HSV values.

In [ ]:
ok, img = grab()
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
h, s, v = np.median(hsv[220:260, 300:340].reshape(-1, 3), axis=0)
print("Hue {:.0f}, saturation {:.0f}, value {:.0f}".format(h, s, v))

## 6. Track a coloured object

The cell keeps the pixels whose HSV is close to your object's, finds the biggest patch,
and draws a circle around it. The LED under the object's position lights up: LED 3 on the
left of the picture, LED 0 on the right. Press **button 0** to stop (or wait 60 seconds).

Set `HUE` to the hue the previous cell printed. Blue objects are around 110, green around 60,
red near 0 or 175.

In [ ]:
HUE = int(h)          # from the cell above, or type a number
lower = np.array([max(HUE - 10, 0), 100, 60])
upper = np.array([min(HUE + 10, 179), 255, 255])

t0 = time.time()
while time.time() - t0 < 60 and not base.buttons[0].read():
    ok, img = grab()
    if not ok:
        break
    mask = cv2.inRange(cv2.cvtColor(img, cv2.COLOR_BGR2HSV), lower, upper)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        biggest = max(contours, key=cv2.contourArea)
        (x, y), r = cv2.minEnclosingCircle(biggest)
        if r > 10:
            cv2.circle(img, (int(x), int(y)), int(r), (0, 255, 0), 3)
            cv2.drawMarker(img, (int(x), int(y)), (0, 0, 255), cv2.MARKER_CROSS, 20, 2)
            base.leds[0:4].write(1 << (3 - min(int(x) // 160, 3)))
        else:
            base.leds[0:4].off()
    else:
        base.leds[0:4].off()
    out = hdmi_out.newframe()
    out[:] = img
    hdmi_out.writeframe(out)
base.leds[0:4].off()

## Clean up

Release the webcam so other notebooks can use it, and close the HDMI output.

In [ ]:
cam.release()
hdmi_out.close()

## Try it yourself

Show the mask instead of the camera picture on HDMI (hint: `cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)`),
and print the object's area with `cv2.contourArea(biggest)`.

In [ ]:
# your code here